This notebook is used to examine, correct, and export the LLM generated features for structured clustering (ppt, group, dehumanizing language, hate speech, stance). Inputs are loaded from `metaphor-detector/results/llm_ann/llm_feature_extractor/`. Results are saved to `analysis/results`.

In [1]:
import pandas as pd
from src.utils import load_json, save_to_json

In [2]:
# collect features
dataset = "tweets_immigration"
ppt_classifications = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_target_ppt_class_qwen.json")["data"]
person_features = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_person_features.json")["data"]
place_features = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_place_features.json")["data"]
thing_features = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_thing_features.json")["data"]

In [3]:
# combine ppt results into df
ppt_class_dicts = [{"id": k, "met_score": v["score"], "source": v["source"]["text"], "target": v["target"]["text"],
                    "sentence": v["sentence"], "target_noun_class": v["annotation"]["noun_classification"]} for k, v in ppt_classifications.items()]

ppt_df = pd.DataFrame.from_dict(ppt_class_dicts)

In [4]:
# print ppt stats
ppt_grouped_df = ppt_df.groupby("target_noun_class").agg(count = ("id", "count"))
ppt_grouped_df

,count
target_noun_class,
person,1209
place,159
thing,1405


In [5]:
# create feature dfs
def get_feature_df(group_key, features):
    features_dicts = [{
        "id": k, "group": v["annotation"][group_key].lower(), "dehumanization": v["annotation"]["dehumanization"],
        "hate_speech": v["annotation"]["hate_speech"], "stance": v["annotation"]["stance"]} 
    for k, v in features.items()]

    return pd.DataFrame.from_dict(features_dicts)

In [6]:
person_features_df = get_feature_df("target_group", person_features)
place_features_df = get_feature_df("place_type", place_features)
thing_features_df = get_feature_df("thing_type", thing_features)

In [7]:
# print person stats
person_grouped_df = person_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
person_grouped_df

,count
group,
blacks,1
children,1
everyone,1
illegals,1
people of color,1
racists,1
americans,2
law enforcement,5
government,38


In [8]:
# CLEAN PERSON GROUPS
accepted_groups = ["immigrants", "non-immigrant u.s. citizens", "politicians", "government", "law enforcement", "other"]
def clean_groups(group_label):
    clean_group_label = group_label.lower().strip()
    if clean_group_label not in accepted_groups:
        clean_group_label = "other"
    return clean_group_label

person_features_df["group"] = person_features_df["group"].apply(clean_groups)
person_grouped_df = person_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
person_grouped_df

,count
group,
law enforcement,5
government,38
politicians,171
non-immigrant u.s. citizens,220
other,227
immigrants,548


In [9]:
stance_grouped_df = person_features_df.groupby("stance").agg(count = ("id", "count")).sort_values("count")
stance_grouped_df

,count
stance,
Anti-immigration,2
Pro-immigration,3
Neutral,5
anti-immigration,38
pro,90
neutral,314
anti,757


In [10]:
accepted_stance = ["pro", "neutral", "anti"]
def extract_stance(dirty_stance):
    for a_s in accepted_stance:
        if a_s in dirty_stance:
            return a_s
    return 'invalid'

def clean_stance(dirty_stance):
    clean_stance = dirty_stance.lower().strip()
    if clean_stance not in accepted_stance:
        clean_stance = extract_stance(clean_stance)
    
    return clean_stance


person_features_df["stance"] = person_features_df["stance"].apply(clean_stance)
stance_grouped_df = person_features_df.groupby("stance").agg(count = ("id", "count")).sort_values("count")
stance_grouped_df

,count
stance,
pro,93
neutral,319
anti,797


In [11]:
dehum_grouped_df = person_features_df.groupby("dehumanization").agg(count = ("id", "count"))
dehum_grouped_df

,count
dehumanization,
False,733
True,476


In [12]:
hs_grouped_df = person_features_df.groupby("hate_speech").agg(count = ("id", "count"))
hs_grouped_df

,count
hate_speech,
False,927
True,282


In [13]:
# now for place features
place_grouped_df = place_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
place_grouped_df

,count
group,
u.s country,1
u.s. city,16
u.s. state,22
unclear,33
non-u.s.,34
u.s. country,53


In [14]:
# clean groups
def clean_place_groups(group_ann):
    return "u.s. country" if group_ann == "u.s country" else group_ann

place_features_df["group"] = place_features_df["group"].apply(clean_place_groups)
# now for place features
place_grouped_df = place_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
place_grouped_df

,count
group,
u.s. city,16
u.s. state,22
unclear,33
non-u.s.,34
u.s. country,54


In [15]:
# now for place features
stance_grouped_df = place_features_df.groupby("stance").agg(count = ("id", "count")).sort_values("count")
stance_grouped_df


,count
stance,
Anti,2
Pro,3
pro,3
Pro-immigration,5
Neutral,15
neutral,21
Anti-immigration,34
anti,76


In [16]:
place_features_df["stance"] = place_features_df["stance"].apply(clean_stance)
stance_grouped_df = place_features_df.groupby("stance").agg(count = ("id", "count")).sort_values("count")
stance_grouped_df

,count
stance,
pro,11
neutral,36
anti,112


In [17]:
hs_grouped_df = place_features_df.groupby("hate_speech").agg(count = ("id", "count"))
hs_grouped_df

,count
hate_speech,
False,102
True,57


In [18]:
dehum_grouped_df = place_features_df.groupby("dehumanization").agg(count = ("id", "count"))
dehum_grouped_df

,count
dehumanization,
False,83
True,76


In [19]:
# and finally for the thing annotations
thing_grouped_df = thing_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
thing_grouped_df

,count
group,
money,83
idea/opinion,95
abstract concept,365
other,401
immigration,461


In [20]:
thing_features_df["stance"] = thing_features_df["stance"].apply(clean_stance)
stance_grouped_df = thing_features_df.groupby("stance").agg(count = ("id", "count")).sort_values("count")
stance_grouped_df

,count
stance,
pro,114
neutral,397
anti,894


In [21]:
hs_grouped_df = thing_features_df.groupby("hate_speech").agg(count = ("id", "count"))
hs_grouped_df

,count
hate_speech,
False,998
True,407


In [22]:
dehum_grouped_df = thing_features_df.groupby("dehumanization").agg(count = ("id", "count"))
dehum_grouped_df

,count
dehumanization,
False,957
True,448


In [23]:
# combine and export
features_df = pd.concat([person_features_df, place_features_df, thing_features_df])
print(f"Len features df: {len(features_df)}, len ppt df: {len(ppt_df)}")

Len features df: 2773, len ppt df: 2773


In [24]:
merged_df = ppt_df.merge(features_df, on="id", how="left").set_index("id").sort_values("met_score")
print(len(merged_df))
merged_df.head()

2773


,met_score,source,target,sentence,target_noun_class,group,dehumanization,hate_speech,stance
id,,,,,,,,,
1078819899915481088_0_30_33_2,0.4,CLEARED,MIGRATION,"'Bre Payton,26yr old San Diego-likely another ...",thing,immigration,True,True,anti
1136667968849567747_1_0_1_2,0.4,lost,I,'I lost a friend from childhood to Assad.',person,other,False,False,neutral
1057340814186201088_0_6_10_2,0.4,take,children,'If someone is here illegally the children the...,person,other,False,False,anti
1119279260257193985_0_1_2_2,0.4,describes,operation,'This operation describes then from soul POV a...,thing,immigration,False,False,neutral
971512021521960962_0_6_7_1,0.4,have,rights,'Pelosi: Millions of illegal aliens have rights.',thing,abstract concept,False,False,pro


In [25]:
# export clean features to json
clean_dicts = merged_df.to_dict(orient='index')
save_to_json(clean_dicts, "results", f"{dataset}_features.json")
print(f"saved to {dataset}_features.json")

saved to tweets_immigration_features.json


In [26]:
# GUN CONTROL RESULTS
# collect features
dataset = "subframes_guns"
ppt_classifications = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_target_ppt_class_qwen.json")["data"]
person_features = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_person_features.json")["data"]
place_features = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_place_features.json")["data"]
thing_features = load_json(f"/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_feature_extractor/{dataset}_thing_features.json")["data"]

In [27]:
# combine ppt results into df
ppt_class_dicts = [{"id": k, "met_score": v["score"], "source": v["source"]["text"], "target": v["target"]["text"],
                    "sentence": v["sentence"], "target_noun_class": v["annotation"]["noun_classification"]} for k, v in ppt_classifications.items()]

ppt_df = pd.DataFrame.from_dict(ppt_class_dicts)

In [28]:
# print ppt stats
ppt_grouped_df = ppt_df.groupby("target_noun_class").agg(count = ("id", "count"))
ppt_grouped_df

,count
target_noun_class,
person,1749
place,167
thing,3084


In [29]:
person_features_df = get_feature_df("target_group", person_features)
place_features_df = get_feature_df("place_type", place_features)
thing_features_df = get_feature_df("thing_type", thing_features)

In [30]:
# print person stats
person_grouped_df = person_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
person_grouped_df

,count
group,
lawmakers,1
judiciary,20
law enforcement,38
gun rights advocates,67
victims of gun violence,75
violent shooters,76
gun control advocates,82
gun owners,85
politicians,265


In [31]:
thing_grouped_df = thing_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
thing_grouped_df

,count
group,
weapon,109
gun violence,190
idea/opinion,337
law/regulation,412
other,912
abstract concept,1124


In [32]:
#combine

In [33]:
place_grouped_df = place_features_df.groupby("group").agg(count = ("id", "count")).sort_values("count")
place_grouped_df

,count
group,
unclear,9
u.s. school,12
other,18
u.s. state,21
non-u.s.,29
u.s. city,29
u.s.,49


In [34]:
features_df = pd.concat([person_features_df, place_features_df, thing_features_df])
print(f"Len features df: {len(features_df)}, len ppt df: {len(ppt_df)}")

Len features df: 5000, len ppt df: 5000


In [35]:
features_df["stance"] = features_df["stance"].apply(clean_stance)
stance_grouped_df = features_df.groupby("stance").agg(count = ("id", "count")).sort_values("count")
stance_grouped_df

,count
stance,
invalid,1
pro,843
anti,992
neutral,3164


In [36]:
dehum_grouped_df = features_df.groupby("dehumanization").agg(count = ("id", "count"))
dehum_grouped_df

,count
dehumanization,
False,4675
True,325


In [37]:
hs_grouped_df = person_features_df.groupby("hate_speech").agg(count = ("id", "count"))
hs_grouped_df

,count
hate_speech,
False,1701
True,48


In [38]:
# export clean features to json\
merged_df = ppt_df.merge(features_df, on="id", how="left").set_index("id").sort_values("met_score")
clean_dicts = merged_df.to_dict(orient='index')
save_to_json(clean_dicts, "results", f"{dataset}_features.json")
print(f"saved to {dataset}_features.json")

saved to subframes_guns_features.json
